Ma a `lepes` egy `while` ciklusba kerül: **ez az ágens**. Az Ollama fusson, mindkét modellel (`gemma4:e4b`, `bge-m3`), 8192 tokenes kontextussal; net kell (Eurostat, KSH). A projekt részhez kell az MNB-csomag mappája (`projekt-mnb`, a Projekt lapról; a KT-sóknak is), legjobb a `notebookok` mappa mellett. Futtasd sorban: **Shift + Enter**. A Copilot kódját olvasd el, mielőtt **Keep**-et nyomsz; ha rossz, **Undo**.

In [ ]:
import importlib.util
import json
import re
import time
import urllib.request
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import pymupdf

import kurzus
from kurzus import chat, embed, kerdez, utolso_hasznalat
from kurzus import ellenoriz, ellenoriz_feltetel, ellenoriz_fuggveny

## 1. Előkészítés: indítsd el most

**Futtasd le most a rész celláit**: a 3–8. alkalom függvényei, és a 7. alkalom ágensindexe (gyorsítótárból pár másodperc; ha nincs meg, kb. 2-2,5 perc).

In [ ]:
# A 3. alkalom függvényei: PDF → bekezdések → darabok. Csak futtasd.
FEJLEC_JELEK = ("INFLÁCIÓS JELENTÉS •", "MAGYAR NEMZETI BANK")
DARABOK_FAJL = Path("data/cache/darabok.json")


def tisztit(szoveg):
    """PDF-ből kinyert szöveg takarítása: elválasztás, pontsorok, sortörések, szóközök."""
    szoveg = re.sub(r"-\n(?=[a-záéíóöőúüű])", "", szoveg)  # sor végi elválasztás: ren-|delkezésére
    szoveg = re.sub(r"\.{4,}", " ", szoveg)                # tartalomjegyzék pontsora
    szoveg = szoveg.replace("\n", " ")
    return re.sub(r"\s+", " ", szoveg).strip()


def nyomtatott_oldalszam(oldal):
    """Az élőfejben álló nyomtatott oldalszám; ha az oldalon nincs élőfej (borító, belső borító), None."""
    elso_sorok = [s.strip() for s in oldal.get_text().splitlines()[:5]]
    if not any("INFLÁCIÓS JELENTÉS •" in s for s in elso_sorok):
        return None
    for sor in elso_sorok:
        if sor.isdigit():
            return int(sor)
    return None


def oldal_bekezdesei(oldal):
    """Az oldal bekezdései takarítva, élőfej, grafikonfeliratok és oldalszám nélkül."""
    bekezdesek = []
    for blokk in oldal.get_text("blocks"):
        szoveg = tisztit(blokk[4])
        if any(jel in szoveg for jel in FEJLEC_JELEK) or szoveg.isupper():
            continue
        if len(szoveg.split()) < 5 or not re.search(r"[a-zA-ZáéíóöőúüűÁÉÍÓÖŐÚÜŰ]{3}", szoveg):
            continue
        bekezdesek.append(szoveg)
    return bekezdesek


def jelentes_oldalai(pdf_utvonal, jelentes):
    """Egy jelentés oldalai: jelentés, PDF-oldalszám, nyomtatott oldalszám és a bekezdések."""
    return [{"jelentes": jelentes, "pdf_oldal": i, "oldal": nyomtatott_oldalszam(o), "bekezdesek": oldal_bekezdesei(o)}
            for i, o in enumerate(pymupdf.open(pdf_utvonal), start=1)]


def darabol_fix(szoveg, meret=1000, atfedes=0):
    """Fix méretű darabok (karakterben); az egymást követő darabok `atfedes` karakternyit átfednek."""
    lepes = meret - atfedes
    return [szoveg[i:i + meret] for i in range(0, max(len(szoveg) - atfedes, 1), lepes)]


ROVIDITESEK = {"pl", "kb", "stb", "ill", "ún", "vö", "szp", "Kr", "dr"}


def mondatokra(szoveg):
    """Mondatokra bontja a szöveget. Mondatvég: pont, kérdőjel vagy felkiáltójel, utána szóköz és
    nagybetű vagy számjegy; kivéve, ha a pont előtt szám, római szám, egyetlen betű vagy rövidítés áll."""
    mondatok = []
    for resz in re.split(r"(?<=[.!?]) (?=[A-ZÁÉÍÓÖŐÚÜŰ0-9])", szoveg):
        elozo_szo = mondatok[-1].split()[-1].rstrip(".") if mondatok else ""
        if elozo_szo.isdigit() or re.fullmatch(r"[IVX]+|\w", elozo_szo) or elozo_szo in ROVIDITESEK:
            mondatok[-1] += " " + resz
        else:
            mondatok.append(resz)
    return mondatok


def darabol_bekezdesenkent(bekezdesek, max_hossz=1000):
    """Egész bekezdéseket fűz össze, amíg a darab el nem érné a `max_hossz` karaktert. A túl hosszú
    bekezdést mondatokra bontja; ha egyetlen mondat is túl hosszú, azt szavakra."""
    reszek = []
    for bekezdes in bekezdesek:
        if len(bekezdes) <= max_hossz:
            reszek.append(bekezdes)
            continue
        for mondat in mondatokra(bekezdes):
            reszek += [mondat] if len(mondat) <= max_hossz else mondat.split()
    darabok, aktualis = [], ""
    for resz in reszek:
        if aktualis and len(aktualis) + 1 + len(resz) > max_hossz:
            darabok.append(aktualis)
            aktualis = resz
        else:
            aktualis = (aktualis + " " + resz).strip()
    if aktualis:
        darabok.append(aktualis)
    return darabok


def darabok_keszitese(oldalak, max_hossz=1000):
    """Egy jelentés darabjai; minden darab megőrzi a jelentést és mindkét oldalszámot."""
    darabok = []
    for o in oldalak:
        for j, szoveg in enumerate(darabol_bekezdesenkent(o["bekezdesek"], max_hossz)):
            darabok.append({"id": f'{o["jelentes"]}_p{o["pdf_oldal"]:03d}_{j}', "jelentes": o["jelentes"],
                            "pdf_oldal": o["pdf_oldal"], "oldal": o["oldal"], "szoveg": szoveg})
    return darabok


def darabok_betoltese(fajl=DARABOK_FAJL):
    """A forrasok.csv összes jelentésének darabjai a gyorsítótárból; ha nincs meg, újra elkészíti és elmenti (kb. 10 mp)."""
    if fajl.exists():
        return json.loads(fajl.read_text(encoding="utf-8"))
    print("Nincs meg a darabok.json: újra elkészítem a 3. alkalom függvényeivel.")
    darabok = []
    for s in pd.read_csv("data/mnb/forrasok.csv", sep=";").itertuples():
        darabok += darabok_keszitese(jelentes_oldalai(kurzus.letolt(s.url, "data/mnb/" + s.fajl), s.jelentes))
    fajl.parent.mkdir(parents=True, exist_ok=True)
    fajl.write_text(json.dumps(darabok, ensure_ascii=False), encoding="utf-8")
    return darabok

In [ ]:
# A 4. alkalom függvényei: beágyazás, index, keresés, ellenőrzés a nyers PDF-ből. Csak futtasd.
INDEX_FAJL = Path("data/cache/embeddingek_rag.npz")    # saját fájl: más jelentések, mint a 4. alkalommal


def beagyaz(szovegek, adag=64):
    """Szövegek embeddingje adagokban (egy hívás = `adag` szöveg); numpy-mátrixot ad vissza."""
    vektorok = []
    for i in range(0, len(szovegek), adag):
        vektorok += embed(szovegek[i:i + adag])
    return np.array(vektorok, dtype=np.float32)


def index_betoltese(darabok, fajl=INDEX_FAJL):
    """A darabok normalizált embedding-mátrixa: a gyorsítótárból, ha ugyanezekhez a darabokhoz
    készült, különben kiszámolja és elmenti. Az i. sor az i. darabé."""
    idk = [d["id"] for d in darabok]
    if fajl.exists():
        tarolt = np.load(fajl)
        if list(tarolt["idk"]) == idk:
            print(f"Gyorsítótárból: {len(idk)} darab (a kiszámolás {float(tarolt['ido']):.0f} mp volt).")
            return tarolt["vektorok"]
    kezdes = time.perf_counter()
    vektorok = beagyaz([d["szoveg"] for d in darabok])
    vektorok = vektorok / np.linalg.norm(vektorok, axis=1, keepdims=True)   # normalizálás
    ido = time.perf_counter() - kezdes
    fajl.parent.mkdir(parents=True, exist_ok=True)
    np.savez(fajl, vektorok=vektorok, idk=np.array(idk), ido=ido)
    print(f"Kiszámolva: {len(idk)} darab, {ido:.0f} mp ({len(idk) / ido:.1f} darab/mp).")
    return vektorok


def keres(kerdes, darabok, index, k=5):
    """A kérdéshez leghasonlóbb k darab, (hasonlóság, darab) párokként, csökkenő sorrendben."""
    q = np.array(embed(kerdes)[0])
    q = q / np.linalg.norm(q)
    hasonlosagok = index @ q                       # koszinusz-hasonlóság az összes darabbal egyszerre
    legjobbak = np.argsort(-hasonlosagok)[:k]      # a k legnagyobb hasonlóság helye
    return [(float(hasonlosagok[i]), darabok[i]) for i in legjobbak]


def kiir(talalatok):
    """A találatok kiírása: hasonlóság, forrás, a szöveg eleje."""
    for hasonlosag, d in talalatok:
        hol = f'{d["oldal"]}. oldal' if d["oldal"] else f'PDF {d["pdf_oldal"]}. oldal'
        print(f'{hasonlosag:.3f}  {d["jelentes"]}, {hol}:  {d["szoveg"][:140]}…')


forrasok = pd.read_csv("data/mnb/forrasok.csv", sep=";")


def oldalak_ahol(minta, jelentesek):
    """(jelentés, PDF-oldal) párok, ahol a nyers PDF-szövegben megvan a minta (reguláris kifejezés)."""
    talalt = set()
    for s in forrasok[forrasok["jelentes"].isin(jelentesek)].itertuples():
        for i, oldal in enumerate(pymupdf.open("data/mnb/" + s.fajl), start=1):
            if re.search(minta, tisztit(oldal.get_text())):
                talalt.add((s.jelentes, i))
    return talalt


def van_jo_talalat(talalatok, vart_oldalak):
    """Igaz, ha a találatok között van olyan darab, amely a várt oldalak egyikéről jön."""
    return any((d["jelentes"], d["pdf_oldal"]) in vart_oldalak for _, d in talalatok)

In [ ]:
# Az 5. alkalom függvényei: a RAG-lánc és az ellenőrzései. Csak futtasd.
HONAPOK = {"01": "január", "02": "február", "03": "március", "04": "április", "05": "május",
           "06": "június", "07": "július", "08": "augusztus", "09": "szeptember", "10": "október",
           "11": "november", "12": "december"}


def forras_cimke(d):
    """Forrásmegjelölés egy darabhoz, például: MNB Inflációs jelentés, 2025. március, 62. oldal."""
    ev, honap = d["jelentes"].split("-")
    hol = f'{d["oldal"]}. oldal' if d["oldal"] else f'PDF {d["pdf_oldal"]}. oldal'
    return f"MNB Inflációs jelentés, {ev}. {HONAPOK[honap]}, {hol}"


def kontextus(talalatok):
    """A modellnek szánt források: sorszám, forrásmegjelölés, aztán a darab szövege."""
    reszek = []
    for i, (hasonlosag, d) in enumerate(talalatok, start=1):
        reszek.append(f"[{i}] {forras_cimke(d)}\n{d['szoveg']}")
    return "\n\n".join(reszek)


def tokenszam(uzenetek):
    """Hány token a kérés a helyi modell szerint. Szöveget vagy üzenetlistát is elfogad.
    Egyetlen tokennyi választ kérünk, gondolkodás nélkül, hogy gyors legyen (1-2 mp)."""
    if isinstance(uzenetek, str):
        uzenetek = [{"role": "user", "content": uzenetek}]
    chat(uzenetek, gondolkodas="none", max_tokens=1)
    return utolso_hasznalat["prompt_tokens"]


RENDSZER = """Az MNB Inflációs jelentéseiből válaszolsz közgazdász hallgatóknak, magyarul, röviden.
Szabályok:
1. Csak a megadott forrásokból válaszolj. Saját tudásból ne írj számot.
2. Minden állítás után szögletes zárójelben jelöld a forrás számát, például [2].
3. Figyelj a jelentés dátumára: a régebbi jelentés előrejelzése nem tényadat. Ha a források eltérnek, a legújabb jelentésből vett tényadatot add meg.
4. Ha a források nem tartalmazzák a választ, ezt írd: "A forrásokban nincs benne." Ilyenkor ne írj számot."""


TOKEN_HATAR = 3000   # a 4096 tokenes ablakból ennyi a kérés; a többi a gondolkodásé és a válaszé


def rag_valasz(kerdes, darabok, index, k=5, rendszer=RENDSZER, gondolkodas="none"):
    """A teljes RAG-lánc: keresés → kontextus → tokenszám → modell. Szótárat ad vissza."""
    talalatok = keres(kerdes, darabok, index, k)
    while True:
        uzenetek = [{"role": "system", "content": rendszer},
                    {"role": "user", "content": f"Források:\n\n{kontextus(talalatok)}\n\nKérdés: {kerdes}"}]
        tokenek = tokenszam(uzenetek)
        if tokenek <= TOKEN_HATAR or len(talalatok) == 1:
            break
        talalatok = talalatok[:-1]           # nem fér bele: a leggyengébb találat kimarad
    print(f"A kérés: {tokenek} token, {len(talalatok)} darab (határ: {TOKEN_HATAR}, ablak: 4096)")
    kezdes = time.perf_counter()
    valasz = chat(uzenetek, gondolkodas=gondolkodas)["content"]
    return {"kerdes": kerdes, "valasz": valasz, "talalatok": talalatok, "tokenek": tokenek,
            "valasz_tokenek": utolso_hasznalat.get("completion_tokens"),
            "ido": time.perf_counter() - kezdes}


def hivatkozasok(valasz):
    """A válaszban szögletes zárójelben álló forrásszámok, növekvő sorrendben, ismétlés nélkül."""
    szamok = set()
    for belseje in re.findall(r"\[([\d,\s]+)\]", valasz):     # [2] vagy [2, 3]
        szamok.update(int(x) for x in re.findall(r"\d+", belseje))
    return sorted(szamok)


def kiir_valasz(eredmeny):
    """A válasz, alatta a hivatkozott források teljes megjelölésével (jelentés, nyomtatott oldal)."""
    print(eredmeny["valasz"])
    print("\nHivatkozott források:")
    for n in hivatkozasok(eredmeny["valasz"]):
        if 1 <= n <= len(eredmeny["talalatok"]):
            print(f"  [{n}] {forras_cimke(eredmeny['talalatok'][n - 1][1])}")
        else:
            print(f"  [{n}] nincs ilyen sorszámú forrás: a modell kitalálta")


_oldal_tar = {}


def oldal_szovege(jelentes, pdf_oldal):
    """Egy PDF-oldal nyers szövege, takarítva. A darabolástól független: ezzel ellenőrzünk."""
    if (jelentes, pdf_oldal) not in _oldal_tar:
        fajl = forrasok.loc[forrasok["jelentes"] == jelentes, "fajl"].iloc[0]
        _oldal_tar[(jelentes, pdf_oldal)] = tisztit(pymupdf.open("data/mnb/" + fajl)[pdf_oldal - 1].get_text())
    return _oldal_tar[(jelentes, pdf_oldal)]


def szam_a_pdfbol(minta, jelentes):
    """A minta zárójeles részei (a keresett számok) az első olyan oldalon, ahol a minta megvan."""
    fajl = forrasok.loc[forrasok["jelentes"] == jelentes, "fajl"].iloc[0]
    for i in range(1, len(pymupdf.open("data/mnb/" + fajl)) + 1):
        talalat = re.search(minta, oldal_szovege(jelentes, i))
        if talalat:
            return talalat.groups()
    return None


def szam_benne(szam, szoveg):
    """Önálló számként szerepel-e a szám a szövegben (a 13,7-ben nincs benne a 3,7). A tizedespont is jó."""
    szoveg = re.sub(r"(\d)\.(\d)", r"\1,\2", szoveg)
    return re.search(rf"(?<![\d,]){re.escape(szam)}(?!\d|,\d)", szoveg) is not None


def alatamasztva(eredmeny, szam):
    """A szám benne van a válaszban, ÉS legalább egy hivatkozott darab nyers PDF-oldalán is."""
    if not szam_benne(szam, eredmeny["valasz"]):
        return False
    for n in hivatkozasok(eredmeny["valasz"]):
        if 1 <= n <= len(eredmeny["talalatok"]):
            d = eredmeny["talalatok"][n - 1][1]
            if szam_benne(szam, oldal_szovege(d["jelentes"], d["pdf_oldal"])):
                return True
    return False


def nem_tudja(valasz):
    """A válasz jelzi, hogy a forrásokban nincs benne, és nem ad meg számot (évszámot kivéve)."""
    jelzi = re.search(r"nincs benne|nem tudom|nem tartalmaz|nem szerepel", valasz.lower()) is not None
    szamok = [s for s in re.findall(r"\d+(?:[,.]\d+)?", valasz) if not re.fullmatch(r"(19|20)\d\d", s)]
    return jelzi and not szamok

In [ ]:
# Az ágensblokk jelentései: az 5-6. alkalom öte, és a 2023. szeptemberi (a mintaprojekt kérdése miatt).
AGENS_JELENTESEK = ["2023-09", "2023-12", "2024-03", "2024-09", "2025-03", "2026-09"]

# A 6. alkalom takarított darabolása (táblázatsorok külön, címkézett mondatként), 6 jelentésre.
def sor_mondat(cellak, evek, cimke):
    """Egy táblázatsorból címkézett mondat. cellak: a sorcím, utána az értékek; evek: a táblázat fejléce."""
    sorcim, ertekek = cellak[0], cellak[1:]
    if len(ertekek) == len(evek):                     # csak akkor párosítjuk az évekkel, ha egyértelmű
        ertekek = [f"{ev}: {ertek}" for ev, ertek in zip(evek, ertekek)]
    eleje = f"{cimke}, {sorcim}" if cimke else sorcim
    return f"{eleje}: " + "; ".join(ertekek)


BETUK = r"[a-zA-ZáéíóöőúüűÁÉÍÓÖŐÚÜŰ]{3}"
SZAMCELLA = r"[\d\s,.\-–−()%]+"            # csak számjegy, vessző, pont, kötőjel, zárójel, százalék


def oldal_bekezdesei_es_sorai(oldal):
    """Az oldal bekezdései (a 3. alkalom szabályával) és a táblázatsorai, címkézett mondatként (sor_mondat)."""
    bekezdesek, sorok = [], []
    cim, csoport, evek = "", "", []
    for blokk in oldal.get_text("blocks"):
        cellak = [c.strip() for c in blokk[4].split("\n") if c.strip()]   # a táblázatban cellánként új sor
        szoveg = tisztit(blokk[4])
        if not cellak or any(jel in szoveg for jel in FEJLEC_JELEK) or szoveg.isupper():
            continue                                        # üres blokk, élőfej, élőláb
        if re.match(r"\d+-\d+\. táblázat:", szoveg):        # táblázatcím: innen új táblázat kezdődik
            cim, csoport, evek = szoveg, "", []
        fejlec = []
        while cellak and re.fullmatch(r"20\d\d", cellak[0]):  # a fejléc évei: 2024, 2025, 2026
            fejlec.append(cellak.pop(0))
        if len(fejlec) >= 2:
            evek = fejlec
            szoveg = tisztit(" ".join(cellak))
        if not cellak:
            continue
        if (len(cellak) >= 2 and re.search(BETUK, cellak[0])
                and all(re.fullmatch(SZAMCELLA, c) for c in cellak[1:])):
            cimke = ", ".join(dict.fromkeys(x for x in [cim, csoport] if x))   # ismétlés nélkül
            sorok.append(sor_mondat(cellak, evek, cimke))  # táblázatsor: szöveg, utána csupa szám
            continue
        if (len(cellak) == 1 and re.search(BETUK, cellak[0]) and not cellak[0].endswith(".")
                and len(cellak[0].split()) <= 10):
            csoport = cellak[0]                             # csoportcím, például: GDP (éves növekedés, %)
        if len(szoveg.split()) >= 5 and re.search(BETUK, szoveg):
            bekezdesek.append(szoveg)                       # a 3. alkalom szabálya
    return bekezdesek, sorok


darabok = []
for s in forrasok[forrasok["jelentes"].isin(AGENS_JELENTESEK)].itertuples():
    for p, oldal_ in enumerate(pymupdf.open("data/mnb/" + s.fajl), start=1):
        bekezdesek, sorok = oldal_bekezdesei_es_sorai(oldal_)
        alap = {"jelentes": s.jelentes, "pdf_oldal": p, "oldal": nyomtatott_oldalszam(oldal_)}
        for i, szoveg in enumerate(darabol_bekezdesenkent(bekezdesek)):
            darabok.append(dict(alap, id=f"{s.jelentes}_p{p:03d}_{i}", szoveg=szoveg))
        for i, szoveg in enumerate(sorok):                  # minden táblázatsor külön darab
            darabok.append(dict(alap, id=f"{s.jelentes}_p{p:03d}_t{i}", szoveg=szoveg))
print(f"{len(darabok)} darab, ebből {sum('_t' in d['id'] for d in darabok)} táblázatsor")

index = index_betoltese(darabok, Path("data/cache/embeddingek_agens.npz"))   # kb. 2-2,5 perc, ha nincs meg

In [ ]:
def keres_jelentesben(kerdes, jelentes, k=5):
    """Keresés csak egy jelentés darabjai között: ha a kérdés megnevezi a jelentést, a többi csak zavar."""
    sorok = [i for i, d in enumerate(darabok) if d["jelentes"] == jelentes]
    return keres(kerdes, [darabok[i] for i in sorok], index[sorok], k)

In [ ]:
# A 8. alkalomból: szám a szövegben, egy eszközhívás végrehajtása, a kért eszközök listája,
# és két eszköz (százalékos változás, mai dátum). Csak futtasd.
def szam_benne(szam, szoveg):
    """Önálló számként szerepel-e a szám a szövegben (a 13,7-ben nincs benne a 3,7). A tizedespont is jó."""
    szoveg = re.sub(r"(\d)\.(\d)", r"\1,\2", szoveg)
    szam = str(szam).replace(".", ",")
    return re.search(rf"(?<![\d,]){re.escape(szam)}(?!\d|,\d)", szoveg) is not None


def eszkoz_uzenet(hivas, eszkozok=None):
    """Egy eszközhívás végrehajtása: a modell kérése (a tool_calls egy eleme) → a válaszüzenet (role: tool)."""
    eszkozok = eszkozok or ESZKOZOK
    nev = hivas["function"]["name"]
    if nev in eszkozok:
        argumentumok = json.loads(hivas["function"]["arguments"] or "{}")   # a modell szövegként küldi
        fuggveny = eszkozok[nev]                                          # a szótárból
        eredmeny = fuggveny(**argumentumok)
    else:
        eredmeny = {"hiba": f"nincs ilyen eszköz: {nev}"}
    return {"role": "tool", "tool_call_id": hivas["id"], "content": json.dumps(eredmeny, ensure_ascii=False)}


def hivott_eszkozok(uzenetek):
    """Az üzenetlistában a modell által kért eszközhívások: (név, argumentumok) párok, sorrendben."""
    hivasok = []
    for u in uzenetek:
        for h in u.get("tool_calls") or []:
            hivasok.append((h["function"]["name"], json.loads(h["function"]["arguments"] or "{}")))
    return hivasok


def szazalekos_valtozas(regi, uj):
    """Hány százalékkal változott az érték a régihez képest, két tizedesre."""
    return round((uj - regi) / regi * 100, 2)


ESZKOZ_VALTOZAS = {
    "type": "function",
    "function": {
        "name": "szazalekos_valtozas",
        "description": "Kiszámolja, hány százalékkal nőtt (vagy csökkent) egy érték, például egy árindex "
                       "vagy egy árfolyam, a régi értékhez képest. Két tizedesre kerekít.",
        "parameters": {
            "type": "object",
            "properties": {
                "regi": {"type": "number", "description": "a kiinduló érték"},
                "uj": {"type": "number", "description": "az új érték"},
            },
            "required": ["regi", "uj"],
        },
    },
}


def mai_datum():
    """A mai dátum ÉÉÉÉ-HH-NN formában."""
    return date.today().isoformat()


ESZKOZ_DATUM = {
    "type": "function",
    "function": {
        "name": "mai_datum",
        "description": "A mai dátum, ÉÉÉÉ-HH-NN formában. Akkor használd, ha a válaszhoz tudni kell, milyen nap van ma.",
        "parameters": {"type": "object", "properties": {}},
    },
}


ESZKOZOK = {"szazalekos_valtozas": szazalekos_valtozas, "mai_datum": mai_datum}   # név → függvény

## 2. Az ágens egy ciklus

A `lepes` a 8. alkalomé, két változtatással: a hosszú eszközeredményt röviden írja ki, és minden lépést feljegyez a `NAPLO` listába (tokenszám, kért eszközök).

In [ ]:
RENDSZER = ("Magyarul válaszolj, röviden. Adatot és számolást csak az eszközökkel szerezz: számot csak az "
            "eszközök eredményéből írj. Ha egy eszköz hibát ad, olvasd el, és javítsd a hívást.")


NAPLO = []   # lépésenként: a kérés tokenszáma és a kért eszközök


def rovid(szoveg, hossz=150):
    """Hosszú eszközeredmény rövidítve, a kiíráshoz (a modell a teljeset kapja)."""
    return szoveg if len(szoveg) <= hossz else szoveg[:hossz] + f"… ({len(szoveg)} karakter)"


def lepes(uzenetek, leirasok):
    """Egy lépés: modellhívás; ha eszközt kér, lefuttatjuk, és az eredményt az előzményekbe írjuk.
    True-t ad, ha a modell szövegesen válaszolt (kész)."""
    valasz = chat(uzenetek, eszkozok=leirasok, gondolkodas="none")
    uzenetek.append(valasz)
    hivasok = valasz.get("tool_calls") or []
    NAPLO.append({"lepes": len(NAPLO) + 1, "tokenek": utolso_hasznalat.get("prompt_tokens", 0),
                  "eszkozok": [h["function"]["name"] for h in hivasok]})
    print(f"{len(NAPLO)}. lépés (a kérés {NAPLO[-1]['tokenek']} token)")
    if not hivasok:
        print("   VÁLASZ ", valasz["content"])
        return True
    for hivas in hivasok:
        uzenet = eszkoz_uzenet(hivas)
        print(f"   ESZKÖZ  {hivas['function']['name']}({hivas['function']['arguments']}) -> {rovid(uzenet['content'])}")
        uzenetek.append(uzenet)
    return False

### Feladat 2.1 – Copilottal

> Írj egy `agens(kerdes, leirasok, max_lepes=6, rendszer=None)` függvényt. Az elején állítsa össze az üzenetlistát: egy `system` üzenet (a `rendszer`, vagy ha az `None`, a `RENDSZER` változó) és egy `user` üzenet (a `kerdes`), és ürítse ki a `NAPLO` listát (`NAPLO.clear()`). Utána egy `while` ciklusban hívja a `lepes(uzenetek, leirasok)` függvényt, amíg az `True`-t nem ad (a modell válaszolt), de legfeljebb `max_lepes`-szer. Ha elérte a korlátot, írja ki, hogy „LÉPÉSKORLÁT”, és álljon meg. Mindkét esetben az üzenetlistát adja vissza. Ne használj `while True`-t.

Olvasd el, mielőtt **Keep**-et nyomsz: hol áll meg a ciklus, ha a modell válaszolt? És ha soha nem válaszol? Az ellenőrzés egy „hamis modellel” próbálja ki, valódi modell nélkül.

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
def _hamis_chat(valasz_lepes):
    """A modell helyett, az ellenőrzéshez: a valasz_lepes-edik hívásra válaszol, addig mindig eszközt kér.
    Ha valasz_lepes None, soha nem válaszol (mint egy elakadt ágens). 20 hívás után leállítja a próbát.
    Hívásonként feljegyzi, látta-e a modell az előző lépés eszközeredményét (az előzmény megmarad-e)."""
    hivasok = []

    def hamis(uzenetek, eszkozok=None, **_):
        elozo = f"proba_{len(hivasok)}"
        elozo_valaszolt = valasz_lepes and len(hivasok) >= valasz_lepes   # az előző hívás nem kért eszközt
        hivasok.append(not hivasok or elozo_valaszolt
                       or any(u.get("role") == "tool" and u.get("tool_call_id") == elozo for u in uzenetek))
        utolso_hasznalat["prompt_tokens"] = 0
        if len(hivasok) > 20:
            raise RuntimeError("végtelen ciklus: 20 hívás után sem állt meg")
        if valasz_lepes and len(hivasok) >= valasz_lepes:
            return {"role": "assistant", "content": "kész"}
        return {"role": "assistant", "content": "", "tool_calls": [
            {"id": f"proba_{len(hivasok)}", "type": "function", "function": {"name": "mai_datum", "arguments": "{}"}}]}
    return hamis, hivasok


def _agens_proba(f):
    """Két próba a valódi modell nélkül: egy soha meg nem álló „modell” (korlát: 3) és egy, amelyik a 2. lépésben válaszol."""
    if not callable(f):
        return False
    eredeti = globals()["chat"]
    try:
        jo = []
        for valasz_lepes, korlat, vart_hivas in [(None, 3, 3), (2, 6, 2)]:
            hamis, hivasok = _hamis_chat(valasz_lepes)
            globals()["chat"] = hamis
            u = f("próbakérdés", [], max_lepes=korlat)
            if not all(hivasok):
                print("A modell a következő lépésben nem látta az előző eszközeredményt: "
                      "az egész ciklus ugyanazt az uzenetek listát adja a lepes-nek, és ezt adja vissza.")
            jo.append(len(hivasok) == vart_hivas and all(hivasok) and u[0]["role"] == "system"
                      and u[1]["content"] == "próbakérdés" and (not valasz_lepes or u[-1].get("content") == "kész"))
        return all(jo)
    except Exception as hiba:
        print("Hiba a próbában:", hiba)
        return False
    finally:
        globals()["chat"] = eredeti


ellenoriz_feltetel("agens", _agens_proba,
                   "a ciklus a válasznál megáll (2 lépés), a lépéskorlátnál is (3 lépés), és az előzmény megmarad")

Az én változatom (ezzel megy tovább a notebook):

In [ ]:
def agens(kerdes, leirasok, max_lepes=6, rendszer=None):
    """Az ágensciklus: lépés lépés után, amíg a modell szövegesen nem válaszol, legfeljebb max_lepes lépésig."""
    uzenetek = [{"role": "system", "content": rendszer or RENDSZER}, {"role": "user", "content": kerdes}]
    NAPLO.clear()
    lepes_szam = 0
    while lepes_szam < max_lepes:
        lepes_szam += 1
        if lepes(uzenetek, leirasok):
            return uzenetek
    print(f"LÉPÉSKORLÁT: {max_lepes} lépés után sincs válasz, megállok.")
    return uzenetek

## 3. Többlépéses kérdés

> Melyik EU-országban nőtt legjobban a GDP 2022 és 2024 között, és mekkora volt ott az infláció 2024-ben?

Két új eszköz az Eurostat adat-API-jából (élőben; ha nincs net, a `data/eu_orszagok.csv`-ből, leírása: `data/eu_orszagok.md`): `gdp_valtozas(tol, ig, n)` (a reál-GDP változása, csökkenő sorrendben) és `inflacio_eu(orszag, ev)` (HICP). Olvasd el: ki számolja a GDP változását?

In [ ]:
EUROSTAT_API = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/"
EU_ORSZAGOK = pd.read_csv("data/eu_orszagok.csv", sep=";")      # a 27 tagállam, és a mentett adat
NEVEK = dict(zip(EU_ORSZAGOK["orszag"], EU_ORSZAGOK["nev"]))      # "HU" → "Magyarország"


def eurostat(tabla, szurok, tol, ig):
    """Egy Eurostat-tábla a 27 EU-országra, élőben: {(országkód, év): érték}. Ha nem éri el, None."""
    geo = "".join(f"&geo={k}" for k in NEVEK)
    url = (f"{EUROSTAT_API}{tabla}?{szurok}{geo}&sinceTimePeriod={tol}&untilTimePeriod={ig}"
           "&format=JSON&lang=en")
    try:
        keres = urllib.request.Request(url, headers={"User-Agent": "kurzus-notebook"})
        with urllib.request.urlopen(keres, timeout=30) as valasz:
            adat = json.loads(valasz.read())
    except Exception as hiba:
        print(f"(Nem érem el az Eurostatot: {type(hiba).__name__}. A mentett adatot használom: data/eu_orszagok.csv)")
        return None
    orszagok = {i: k for k, i in adat["dimension"]["geo"]["category"]["index"].items()}
    evek = {i: int(t) for t, i in adat["dimension"]["time"]["category"]["index"].items()}
    # Az értékek egy lapos listában jönnek: sorszám = ország sorszáma × évek száma + év sorszáma.
    return {(orszagok[int(n) // len(evek)], evek[int(n) % len(evek)]): v for n, v in adat["value"].items()}


def mentett(oszlop):
    """Ugyanez a mentett táblából (data/eu_orszagok.csv), ha nincs net."""
    sorok = EU_ORSZAGOK.dropna(subset=[oszlop])
    return {(s.orszag, s.ev): getattr(s, oszlop) for s in sorok.itertuples()}


def gdp_valtozas(tol, ig, n=5):
    """A reál-GDP változása (%) két év között az EU-országokban, a legnagyobbtól kezdve; az első n ország."""
    tol, ig = int(tol), int(ig)
    adat, honnan = eurostat("nama_10_gdp", "na_item=B1GQ&unit=CLV20_MEUR", tol, ig), "élő"
    if adat is None:
        adat, honnan = mentett("gdp_volumen"), "mentett tartalék"
    valtozas = {k: round((adat[(k, ig)] / adat[(k, tol)] - 1) * 100, 2)
                for k in NEVEK if (k, tol) in adat and (k, ig) in adat}
    if not valtozas:
        return {"hiba": f"nincs GDP-adat erre: {tol}–{ig}"}
    rangsor = sorted(valtozas.items(), key=lambda par: par[1], reverse=True)[:int(n)]
    return {"mutato": f"reál-GDP változása {tol} és {ig} között, %", "forras": "Eurostat nama_10_gdp",
            "adat_honnan": honnan, "rangsor": [{"orszag": k, "nev": NEVEK[k], "valtozas": v} for k, v in rangsor]}


def inflacio_eu(orszag, ev):
    """Egy EU-ország éves átlagos inflációja (HICP, %) egy évben."""
    ev = int(ev)
    adat, honnan = eurostat("prc_hicp_ainr", "coicop18=TOTAL&unit=RCH_A_AVG", ev, ev), "élő"
    if adat is None:
        adat, honnan = mentett("hicp"), "mentett tartalék"
    if (orszag, ev) not in adat:
        return {"hiba": f"nincs adat: {orszag}, {ev}; az ország kétbetűs kódja kell, pl. HU, MT"}
    return {"orszag": orszag, "ev": ev, "hicp": adat[(orszag, ev)], "forras": "Eurostat prc_hicp_ainr",
            "adat_honnan": honnan}


ESZKOZ_GDP = {
    "type": "function",
    "function": {
        "name": "gdp_valtozas",
        "description": "Az EU-tagállamok reál-GDP-jének változása (%) két év között (a tol év szintjétől az ig év "
                       "szintjéig), csökkenő sorrendben; az első n országot adja, kétbetűs kóddal és névvel.",
        "parameters": {
            "type": "object",
            "properties": {
                "tol": {"type": "integer", "description": "a kiinduló év, pl. 2022"},
                "ig": {"type": "integer", "description": "a záró év, pl. 2024"},
                "n": {"type": "integer", "description": "hány országot adjon vissza (alapértelmezés: 5)"},
            },
            "required": ["tol", "ig"],
        },
    },
}

ESZKOZ_INFLACIO_EU = {
    "type": "function",
    "function": {
        "name": "inflacio_eu",
        "description": "Egy EU-tagállam éves átlagos inflációja (HICP, %) egy adott évben, az Eurostattól.",
        "parameters": {
            "type": "object",
            "properties": {
                "orszag": {"type": "string", "description": "az ország kétbetűs Eurostat-kódja, pl. HU, MT, DE"},
                "ev": {"type": "integer", "description": "az év, pl. 2024"},
            },
            "required": ["orszag", "ev"],
        },
    },
}

ESZKOZOK.update({"gdp_valtozas": gdp_valtozas, "inflacio_eu": inflacio_eu})

### Feladat 3.1 – olvasás

Hány lépésben (modellhívásban) válaszol az ágens erre a kérdésre? Egy lépés = egy modellhívás; ha a modell egy lépésben több eszközt kér, az is egy lépés. Írd be: `2`, `3` vagy `4`.

In [ ]:
tipp_3_1 = None  # a None helyére írd a tipped

In [ ]:
KERDES_GDP = "Melyik EU-országban nőtt legjobban a GDP 2022 és 2024 között, és mekkora volt ott az infláció 2024-ben?"
EU_ESZKOZOK = [ESZKOZ_VALTOZAS, ESZKOZ_DATUM, ESZKOZ_GDP, ESZKOZ_INFLACIO_EU]

u_gdp = agens(KERDES_GDP, EU_ESZKOZOK)
naplo_gdp = pd.DataFrame(NAPLO)          # a lépések naplója, a 4. részhez

Az ellenőrzés a mentett táblából számol, és a nyomvonalat is nézi:

In [ ]:
ellenoriz("tipp_3_1", vart=len(naplo_gdp))

gdp = EU_ORSZAGOK.pivot(index="orszag", columns="ev", values="gdp_volumen")
valtozas = ((gdp[2024] / gdp[2022] - 1) * 100).round(2).sort_values(ascending=False)
nyertes = valtozas.index[0]
nyertes_hicp = EU_ORSZAGOK.set_index(["orszag", "ev"]).loc[(nyertes, 2024), "hicp"]
print(f"A mentett tábla szerint: {NEVEK[nyertes]} ({nyertes}), {valtozas.iloc[0]}%; HICP 2024: {nyertes_hicp}%")

eszkoz_eredmenyek = " ".join(u["content"] for u in u_gdp if u["role"] == "tool")
ellenoriz_feltetel("u_gdp", lambda u: NEVEK[nyertes] in u[-1]["content"] and szam_benne(nyertes_hicp, u[-1]["content"]),
                   f"a válaszban {NEVEK[nyertes]} és a 2024-es infláció ({nyertes_hicp}) áll")
hivasok_gdp = hivott_eszkozok(u_gdp)
inflacio_hivas = ("inflacio_eu", {"orszag": nyertes, "ev": 2024})
ellenoriz_feltetel("u_gdp", lambda u: inflacio_hivas in hivasok_gdp and ("gdp_valtozas" in [n for n, _ in hivasok_gdp[:hivasok_gdp.index(inflacio_hivas)]])
                   and szam_benne(valtozas.iloc[0], eszkoz_eredmenyek) and szam_benne(nyertes_hicp, eszkoz_eredmenyek),
                   f"a nyomvonalban: a GDP-rangsor eszközből jött, utána az inflacio_eu({nyertes}, 2024) hívás, és a {nyertes_hicp} is eszközből")

Hány lépés volt? Melyik lépés épült az előzőre? Mit tudott volna egy munkafolyamat?

### Feladat 3.2 – olvasás

Ugyanez a kérdés, de `max_lepes=2`. Mi történik?

- `"a"`: az ágens a GDP-részre válaszol, az inflációt kihagyja;
- `"b"`: az ágens eszközöket hív, aztán válasz nélkül megáll;
- `"c"`: hibát dob.

In [ ]:
tipp_3_2 = None  # "a", "b" vagy "c"

In [ ]:
u_korlat = agens(KERDES_GDP, EU_ESZKOZOK, max_lepes=2)

valaszolt = u_korlat[-1]["role"] == "assistant" and not u_korlat[-1].get("tool_calls")
ellenoriz("tipp_3_2", vart="a" if valaszolt else "b")

Mire jó a lépéskorlát, és mi az ára?

## 4. Az előzmény mint memória

Minden lépésben a teljes eddigi beszélgetés újra elmegy. A 3.1-es futás naplója:

In [ ]:
naplo_gdp      # a 3.1-es futás naplója (a 3.2 már felülírta a NAPLO-t)

### Feladat 4.1 – Copilottal

> A `naplo_gdp` táblázat `tokenek` oszlopában az ágens lépéseinek kérésmérete áll (tokenben), sorrendben. Számold ki, átlagosan hány tokennel nőtt a kérés egyik lépésről a következőre, egy tizedesre kerekítve, és tedd a `novekedes` változóba.

Olvasd el: hány különbség van, ha három lépés volt?

In [ ]:
# Ide kerül a Copilot kódja

In [ ]:
t = naplo_gdp["tokenek"]
ellenoriz("novekedes", vart=round((t.iloc[-1] - t.iloc[0]) / (len(t) - 1), 1))
if len(t) > 1:
    lepesek = (6000 - t.iloc[0]) / ((t.iloc[-1] - t.iloc[0]) / (len(t) - 1))
    print(f"Ilyen ütemben kb. {lepesek:.0f} lépés férne bele 6000 tokenbe.")

Mi nő lépésről lépésre, és miért baj? Mi történik a 8192 tokenes ablak végén?

## 5. Az ágens körüli keret (*harness*)

Memória, eszközök, korlátok, kiértékelés. A kiértékelés az **utat** is nézi (*trajectory*): mit kért az ágens, mit kapott, honnan jött a szám.

In [ ]:
def nyomvonal(uzenetek):
    """Az ágens útja lépésenként: mit kért, és mit kapott vissza (ezt nézi a kiértékelés is)."""
    for u in uzenetek:
        if u["role"] == "assistant" and u.get("tool_calls"):
            print("KÉR    ", ", ".join(f"{h['function']['name']}({h['function']['arguments']})" for h in u["tool_calls"]))
        elif u["role"] == "tool":
            print("KAP    ", rovid(u["content"], 100))
        elif u["role"] == "assistant":
            print("VÁLASZ ", rovid(u["content"] or "", 100))


nyomvonal(u_gdp)

### Feladat 5.1 – hibakeresés

A Copilot ellenőrzése: egy szám tényleg egy eszköz eredményéből jött-e? Próbáld ki a 8. alkalom esetén (a 17,6 a kérdésben volt, eszközhívás nem volt):

In [ ]:
def eszkozbol_jott_v1(szam, uzenetek):
    """Igaz, ha a szám egy eszköz eredményéből jött."""
    return any(szam_benne(szam, u.get("content") or "") for u in uzenetek)


eset_kerdesbol = [{"role": "system", "content": RENDSZER},
                  {"role": "user", "content": "Az MNB 2023-ra 17,6–18,1%-ot várt. Mennyi lett?"},
                  {"role": "assistant", "content": "A sáv alja 17,6%, a tényadatot nem tudom."}]
eset_eszkozbol = [{"role": "system", "content": RENDSZER},
                  {"role": "user", "content": "Mennyi lett az infláció 2023-ban?"},
                  {"role": "assistant", "content": "", "tool_calls": [{"id": "h1", "type": "function",
                      "function": {"name": "inflacio", "arguments": '{"forras": "KSH", "tol": "2023"}'}}]},
                  {"role": "tool", "tool_call_id": "h1", "content": '{"adatok": {"2023": 17.6}}'},
                  {"role": "assistant", "content": "17,6% volt."}]
print("a kérdésből:", eszkozbol_jott_v1(17.6, eset_kerdesbol), "  az eszközből:", eszkozbol_jott_v1(17.6, eset_eszkozbol))

Mit néz meg a függvény? Írd meg jól: `eszkozbol_jott(szam, uzenetek)` csak az eszközök eredményében (`role: "tool"`) keresse a számot.

In [ ]:
# Ide kerül a javítás

In [ ]:
ellenoriz_feltetel("eszkozbol_jott",
                   lambda f: callable(f) and not f(17.6, eset_kerdesbol) and f(17.6, eset_eszkozbol)
                             and not f(17.6, eset_eszkozbol[:3] + eset_eszkozbol[4:]),
                   "a kérdésben vagy a válaszban álló szám nem számít; csak az eszköz eredménye")

Miért volt mindig igaz? Az én változatom (megadható, melyik eszköz eredményében keressen):

In [ ]:
def eszkozbol_jott(szam, uzenetek, eszkoz=None):
    """Igaz, ha a szám egy eszköz eredményében szerepel (ha az eszköz neve meg van adva, annak az eredményében)."""
    hivo = {h["id"]: h["function"]["name"] for u in uzenetek for h in (u.get("tool_calls") or [])}
    return any(szam_benne(szam, u["content"]) for u in uzenetek
               if u["role"] == "tool" and (eszkoz is None or hivo.get(u["tool_call_id"]) == eszkoz))

## 6. A projektben: az ágens a csomag adatlekérő függvényével

> A 2023. szeptemberi MNB-jelentés mekkora éves inflációt várt 2023-ra, és mennyi lett?

Három eszköz: `dokumentum_kereses` (a 7. alkalom indexe), `inflacio` (**az MNB-csomag függvénye**, változtatás nélkül) és `osszevet` (a sáv-összevetés kódban). Az MNB-csomag a KT-sóknak is kell (Projekt lap, `mnb.zip`, a `notebookok` mappa mellé kicsomagolva). Ha a cella nem találja, írd be a `projekt-mnb` mappád teljes útvonalát a `CSOMAG_MAPPA`-ba.

In [ ]:
import importlib.util

CSOMAG_MAPPA = None   # ha a cella nem találja a csomagot: a projekt-mnb mappád útvonala, pl. "../projekt-mnb"


def csomag_betoltese(mappa=None):
    """A projektcsomag adatlekeres.py-ja modulként. Ha nincs megadva mappa, a szokásos helyeken keresi:
    a notebookok mappa mellett, és a saját mappádban, a Dokumentumok, az Asztal és a Letöltések mappában."""
    otthon = ["~/projekt-mnb", "~/Documents/projekt-mnb", "~/Desktop/projekt-mnb", "~/Downloads/projekt-mnb"]
    jeloltek = [mappa] if mappa else ["../projekt-mnb", "../../projekt-mnb", "projekt-mnb", *otthon, "../csomagok/mnb"]
    for j in jeloltek:
        fajl = Path(j).expanduser() / "adatlekeres.py"
        if fajl.exists():
            spec = importlib.util.spec_from_file_location("adatlekeres", fajl)
            modul = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(modul)
            print("A csomag:", fajl.resolve().parent)
            return modul
    raise FileNotFoundError("Nem találom az MNB-csomag adatlekeres.py-ját. Írd be a CSOMAG_MAPPA-ba a projekt-mnb "
                            "mappád teljes útvonalát (VS Code: jobb klikk a mappán, Copy Path; Windowson r\"C:\\...\" "
                            "formában). Ha még nincs meg: töltsd le az mnb.zip-et a kurzusoldal Projekt lapjáról, "
                            "és csomagold ki a notebookok mappa mellé.")


adatlekeres = csomag_betoltese(CSOMAG_MAPPA)

In [ ]:
def dokumentum_kereses(kerdes, jelentes=None):
    """Részletek az MNB Inflációs jelentéseiből (a 7. alkalom indexe), forrásmegjelöléssel; k = 3."""
    talalatok = keres_jelentesben(kerdes, jelentes, k=3) if jelentes else keres(kerdes, darabok, index, k=3)
    return kontextus(talalatok)


def osszevet(ertek, also, felso):
    """Egy érték helye egy sávhoz képest: alatta, benne vagy fölötte. Az összevetést kód végzi, nem a modell."""
    if ertek < also:
        helyzet = "a sáv alatt"
    elif ertek > felso:
        helyzet = "a sáv fölött"
    else:
        helyzet = "a sávon belül"
    return {"ertek": ertek, "sav": [also, felso], "helyzet": helyzet}


ESZKOZ_DOKUMENTUM = {
    "type": "function",
    "function": {
        "name": "dokumentum_kereses",
        "description": "Részleteket keres az MNB Inflációs jelentéseiben (előrejelzések, állítások), forrással "
                       "(jelentés, oldal). Ha a kérdés megnevez egy jelentést, add meg a dátumát.",
        "parameters": {
            "type": "object",
            "properties": {
                "kerdes": {"type": "string", "description": "mit keressen, egy mondatban"},
                "jelentes": {"type": "string", "description": "a jelentés dátuma ÉÉÉÉ-HH formában, pl. 2023-09"},
            },
            "required": ["kerdes"],
        },
    },
}

ESZKOZ_INFLACIO = {
    "type": "function",
    "function": {
        "name": "inflacio",
        "description": "Magyarország tényleges inflációja (%) a KSH (fogyasztóiár-index) vagy az Eurostat "
                       "(HICP) adataiból. Éves átlaghoz az időszak 'ÉÉÉÉ', havi adathoz 'ÉÉÉÉ-HH'.",
        "parameters": {
            "type": "object",
            "properties": {
                "forras": {"type": "string", "enum": ["KSH", "Eurostat"], "description": "az adatforrás"},
                "tol": {"type": "string", "description": "az első időszak, pl. '2023' vagy '2023-08'"},
                "ig": {"type": "string", "description": "az utolsó időszak (ha több időszak kell)"},
            },
            "required": ["forras", "tol"],
        },
    },
}

ESZKOZ_OSSZEVET = {
    "type": "function",
    "function": {
        "name": "osszevet",
        "description": "Megmondja, hogy egy érték egy sávon belül, alatta vagy fölötte van-e.",
        "parameters": {
            "type": "object",
            "properties": {
                "ertek": {"type": "number", "description": "a vizsgált érték, pl. a tényadat"},
                "also": {"type": "number", "description": "a sáv alsó széle"},
                "felso": {"type": "number", "description": "a sáv felső széle"},
            },
            "required": ["ertek", "also", "felso"],
        },
    },
}

ESZKOZOK.update({"dokumentum_kereses": dokumentum_kereses, "inflacio": adatlekeres.inflacio, "osszevet": osszevet})
PROJEKT = [ESZKOZ_DOKUMENTUM, ESZKOZ_INFLACIO, ESZKOZ_OSSZEVET]

A várt értékek a nyers PDF-ből és a KSH táblájából:

In [ ]:
MINTA = "A 2023. szeptemberi MNB-jelentés mekkora éves inflációt várt 2023-ra, és mennyi lett?"
vart_sav = [float(x.replace(",", ".")) for x in
            szam_a_pdfbol(r"Az infláció 2023-ban (\d+,\d) – (\d+,\d) százalék között alakulhat", "2023-09")]
inflacio_tabla = pd.read_csv("data/inflacio_eves.csv", sep=";")
ksh_2023 = float(inflacio_tabla.loc[inflacio_tabla["ev"] == 2023, "ksh_fogyasztoiar"].iloc[0])
print(f"Előrejelzés (PDF): {vart_sav[0]}–{vart_sav[1]}%, tényadat (KSH): {ksh_2023}%")

Először az eddigi rendszerprompttal:

In [ ]:
u_projekt_1 = agens(MINTA, PROJEKT)
print("\nA kért eszközök:", [nev for nev, _ in hivott_eszkozok(u_projekt_1)])

### Feladat 6.1 – olvasás

Nézd meg a nyomvonalat: lekérte az ágens a tényadatot (`inflacio`)? Írd be: `"igen"` vagy `"nem"`.

In [ ]:
tipp_6_1 = None  # "igen" vagy "nem"

In [ ]:
ellenoriz("tipp_6_1", vart="igen" if "inflacio" in [n for n, _ in hivott_eszkozok(u_projekt_1)] else "nem")

Mit tett az ágens, és miért nem elég? A javítás a rendszerpromptban:

In [ ]:
RENDSZER_PROJEKT = (RENDSZER + " Az MNB-jelentések állításait a dokumentum_kereses eszközzel keresd meg. "
                    "Hogy mi lett belőlük (a tényadatot), azt a jelentés nem tudja: azt az inflacio eszközzel kérd le, "
                    "és az összevetést az osszevet eszközzel végezd.")

u_projekt = agens(MINTA, PROJEKT, rendszer=RENDSZER_PROJEKT)

In [ ]:
nyomvonal(u_projekt)
print("\nTokenek lépésenként:", [s["tokenek"] for s in NAPLO])

hivasok = hivott_eszkozok(u_projekt)
ellenoriz_feltetel("u_projekt", lambda u: any(n == "inflacio" and a.get("forras") == "KSH" and str(a.get("tol")) == "2023"
                                              for n, a in hivasok),
                   "a nyomvonalban: inflacio, KSH, 2023 (az MNB a fogyasztóiár-indexet jelzi előre)")
ellenoriz_feltetel("u_projekt", lambda u: eszkozbol_jott(ksh_2023, u, "inflacio")
                                          and all(eszkozbol_jott(x, u, "dokumentum_kereses") for x in vart_sav),
                   f"a tényadat ({ksh_2023}) az inflacio eszközből, a sáv ({vart_sav[0]}–{vart_sav[1]}) a keresésből jött")
ellenoriz_feltetel("u_projekt", lambda u: all(szam_benne(x, u[-1]["content"]) for x in vart_sav + [ksh_2023])
                                          and "belül" in u[-1]["content"],
                   "a válaszban a sáv, a tényadat és a „belül” áll")
ellenoriz_feltetel("NAPLO", lambda n: max(s["tokenek"] for s in n) < 6000, "minden kérés 6000 token alatt maradt")

Hány lépés volt, melyik lépésnél nőtt legjobban a kérés? Mit ellenőriztünk: a választ, vagy az utat is?

### Otthonra

A projekted repójában (párban: közösen), a 8. alkalom `eszkozok.py`-jára építve:

1. írj egy `agens.py`-t: a `lepes`, az `agens` és a `nyomvonal` ebből a notebookból, az `ESZKOZOK` szótárban a csomagod adatlekérő függvénye a 8. alkalommal megírt leírással, és a `dokumentum_kereses` a saját RAG-láncodra;
2. futtasd a 10 tesztkérdésed közül a **3 adatos és a 3 kombinált** kérdést; mindegyik nyomvonalát mentsd a `nyomvonalak.md`-be (mit kért az ágens, mit kapott, hány lépés, hány token);
3. minden kérdéshez írj ellenőrzést: a nyomvonalban a jó eszköz, a jó argumentumokkal (`hivott_eszkozok`), és a végső szám egy eszközből jött (`eszkozbol_jott`), és egyezik az `ellenorzes.py` számával;
4. ha az ágens túl korán megállt vagy rossz eszközt választott, javítsd a rendszerpromptban vagy az eszközleírásban, és írd le a `feltetelezesek.md`-be, mit változtattál.

**KT-csomag:** az eszköz a `kormanyzati_adat(mutato, tol, ig, mertekegyseg)`. A leírásában legyen `enum` a `mutato`-ra (`"egyenleg"`, `"adossag"`) és a `mertekegyseg`-re (`"gdp_szazalek"`, `"millio_ft"`), és mondja ki: a negatív egyenleg hiány, és a `millio_ft` millió forint (a KT-dokumentumok milliárdban írnak).